<a href="https://colab.research.google.com/github/ttktjmt/mjswan/blob/main/examples/colab/anymal_c_velocity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🦢 mjswan Tutorial with anymal_c_velocity**

This notebook is an example of visualization with mjswan for the trained policy with mjlab [anymal_c_velocity](https://github.com/mujocolab/anymal_c_velocity)

<img src="https://github.com/mujocolab/anymal_c_velocity/blob/main/teaser.gif?raw=true" width="300">

## **⚙️ Setup**

In [ ]:
!uv pip install "git+https://github.com/mujocolab/anymal_c_velocity.git" "git+https://github.com/ttktjmt/mjswan.git" --system --prerelease=allow -q

## **✔️ Sanity Check (optional)**

In [ ]:
import re
import subprocess
import sys

process = subprocess.Popen(
    [
        "uv",
        "run",
        "play",
        "Mjlab-Velocity-Flat-Anymal-C",
        "--agent",
        "random",
        "--num_envs",
        "4",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
)

detected_port = None

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

    # Extract port number from viser output
    port_match = re.search(r":(\d{4})", line)
    if port_match and "viser" in line.lower():
        detected_port = int(port_match.group(1))
        print("\n" + "=" * 34)
        print(f"✅ Server is running on port {detected_port}!")
        print("=" * 34)
        break

In [ ]:
from google.colab import output

port = detected_port if detected_port else 8081
output.serve_kernel_port_as_iframe(port=port, height=600)

## **📚 Train the anymal_c_velocity Policy**

### **🔑 WandB Setup**

In [ ]:
# Online WandB
!wandb login

# Local WandB
# !wandb offline

### **🏋️ Train the policy**

This will take a couple of hours

In [ ]:
!uv run train Mjlab-Velocity-Flat-Anymal-C --env.scene.num-envs 4096 --agent.max-iterations 2_001

## **♺ Helper Functions**

In [ ]:
import json
import tempfile
from pathlib import Path

import mujoco
import onnx
from mjlab.scene import Scene
from mjlab.tasks.registry import load_env_cfg


def get_mjlab_scene_spec(task_id: str) -> mujoco.MjSpec:
    env_cfg = load_env_cfg(task_id)
    env_cfg.scene.num_envs = 1
    scene = Scene(env_cfg.scene, device="cpu")
    return scene.spec


def get_onnx_policy(run) -> onnx.ModelProto:
    onnx_file = next((f for f in run.files() if f.name.endswith(".onnx")), None)
    if onnx_file is None:
        raise RuntimeError("No .onnx file found")
    tmp = tempfile.mkdtemp()
    onnx_path = Path(onnx_file.download(root=tmp, replace=True).name)
    print(f"Downloading {onnx_file.name} to {str(onnx_path)}")
    return onnx.load(str(onnx_path))


def get_policy_config(onnx_model: onnx.ModelProto) -> str:
    def parse_floats(s: str) -> list[float]:
        return [float(v.strip()) for v in s.split(",")]

    meta = {prop.key: prop.value for prop in onnx_model.metadata_props}

    # ONNX metadata uses bare joint names (e.g. "LF_HAA"); re-add "robot/" prefix.
    joint_names = [f"robot/{v.strip()}" for v in meta["joint_names"].split(",")]
    default_joint_pos = parse_floats(meta["default_joint_pos"])
    joint_stiffness = parse_floats(meta["joint_stiffness"])
    joint_damping = parse_floats(meta["joint_damping"])
    action_scale_raw = meta["action_scale"]
    try:
        action_scale: float | list[float] = float(action_scale_raw)
    except ValueError:
        action_scale = parse_floats(action_scale_raw)

    initializer_names = {init.name for init in onnx_model.graph.initializer}
    in_keys = [
        inp.name for inp in onnx_model.graph.input if inp.name not in initializer_names
    ]

    # Normalize "actions" → "action" to match mjswan runtime convention.
    _OUT_KEY_MAP = {"actions": "action"}
    out_keys = [_OUT_KEY_MAP.get(out.name, out.name) for out in onnx_model.graph.output]

    obs_config = {
        in_keys[0]: [
            {"name": "BaseLinearVelocity", "world_frame": False},
            {"name": "BaseAngularVelocity"},
            {"name": "ProjectedGravityB"},
            {
                "name": "JointPos",
                "pos_steps": [0],
                "subtract_default": True,
                "scale": 1.0,
            },
            {"name": "JointVelocities"},
            {"name": "PrevActions", "history_steps": 1},
            {"name": "SimpleVelocityCommand"},
        ]
    }

    config = {
        "policy_joint_names": joint_names,
        "default_joint_pos": default_joint_pos,
        "obs_config": obs_config,
        "action_scale": action_scale,
        "stiffness": joint_stiffness,
        "damping": joint_damping,
        "control_type": "joint_position",
        "onnx": {"meta": {"in_keys": in_keys, "out_keys": out_keys}},
    }

    config_path = Path(tempfile.mkdtemp()) / "anymal_c_velocity.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    return str(config_path)

## **🦢 Generate the mjswan App**

In [ ]:
import wandb

import mjswan

run_path = "ttktjmt-org/mjlab/dqxvf0eb"  # Replace with your actual run path if not using the online API
task_id = "Mjlab-Velocity-Flat-Anymal-C"

if wandb.run is not None:
    run = wandb.run
    print(run.name)
else:
    print("Not logged into WandB")
    api = wandb.Api()
    run = api.run(run_path)

APP_DIR = Path("/content/mjswan/app")


builder = mjswan.Builder()
project = builder.add_project(name="anymal_c_velocity")

anymal_c_scene = project.add_scene(spec=get_mjlab_scene_spec(task_id), name=task_id)

policy = get_onnx_policy(run)

anymal_c_scene.add_policy(
    name="anymal_c_velocity",
    policy=policy,
    config_path=get_policy_config(policy),
).add_velocity_command(
    lin_vel_x=(-1.0, 1.0),
    lin_vel_y=(-1.0, 1.0),
    ang_vel_z=(-0.5, 0.5),
    default_lin_vel_x=0.5,
    default_lin_vel_y=0.0,
    default_ang_vel_z=0.0,
)

app = builder.build(output_dir=APP_DIR)

## **🌐 Serve Visualization**

In [ ]:
import http.server
import socketserver
import threading

PORT = 8000


class Handler(http.server.SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=APP_DIR, **kwargs)


def start_server():
    with socketserver.TCPServer(("", PORT), Handler) as httpd:
        print(f"Serving at port {PORT}")
        httpd.serve_forever()


thread = threading.Thread(target=start_server, daemon=True)
thread.start()

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(PORT, height="700")